In [ ]:
import os, sys, glob, time, subprocess
import numpy as np, cv2, torch
import matplotlib.pyplot as plt
from google.colab import drive, userdata, files

drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), 'Drive is not mounted'
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')   # never printed
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'kaggle', 'timm'], check=True)

print('--- KITTI candidates on Kaggle ---')
!kaggle datasets list -s "kitti object detection" --sort-by votes
!kaggle datasets list -s "kitti 3d" --sort-by votes

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('\ndevice:', dev)
clip = '/content/drive/MyDrive/adp_project/videos/clip2.mp4'
assert os.path.exists(clip), 'clip2.mp4 missing in Drive videos folder'
cap = cv2.VideoCapture(clip); cap.set(cv2.CAP_PROP_POS_FRAMES, 60); ok, frame = cap.read(); cap.release()
assert ok, 'could not read a frame'
img = cv2.cvtColor(cv2.resize(frame, (1280, 720)), cv2.COLOR_BGR2RGB)

how, depth, ms = None, None, None
try:
    midas = torch.hub.load('intel-isl/MiDaS', 'MiDaS_small', trust_repo=True).to(dev).eval()
    tf = torch.hub.load('intel-isl/MiDaS', 'transforms', trust_repo=True).small_transform
    x = tf(img).to(dev)
    with torch.no_grad():
        for _ in range(3): midas(x)
        if dev == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(20): out = midas(x)
        if dev == 'cuda': torch.cuda.synchronize()
        ms = (time.perf_counter() - t0) / 20 * 1000
    depth = torch.nn.functional.interpolate(out.unsqueeze(1), size=img.shape[:2], mode='bicubic',
                                            align_corners=False).squeeze().cpu().numpy()
    how = 'torch.hub intel-isl/MiDaS MiDaS_small'
except Exception as e:
    print('torch.hub route failed:', type(e).__name__, str(e)[:250])

if how is None:
    try:
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'transformers'], check=True)
        from transformers import pipeline
        from PIL import Image
        pipe = pipeline('depth-estimation', model='Intel/dpt-hybrid-midas', device=0 if dev == 'cuda' else -1)
        out = pipe(Image.fromarray(img))
        depth = np.asarray(out['predicted_depth'].squeeze().cpu() if hasattr(out['predicted_depth'], 'cpu') else out['predicted_depth'])
        depth = cv2.resize(depth, (img.shape[1], img.shape[0]))
        how = 'Hugging Face Intel/dpt-hybrid-midas (fallback)'
    except Exception as e:
        print('Hugging Face route failed:', type(e).__name__, str(e)[:250])

assert how is not None, 'Neither MiDaS route worked - send me the two error lines above'
print('\nMiDaS route:', how)
print('depth map shape:', depth.shape, '| min/max:', float(depth.min()), float(depth.max()))
if ms: print(f'MiDaS_small forward pass: {ms:.1f} ms (device {dev}, input 256 on the short side by default)')
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(img); ax[0].set_title('frame'); ax[0].axis('off')
ax[1].imshow(depth, cmap='inferno'); ax[1].set_title('relative depth (brighter = closer)'); ax[1].axis('off')
plt.tight_layout(); plt.savefig('/content/depth_check.png', dpi=110); plt.show()
files.download('/content/depth_check.png')

In [ ]:
import subprocess, sys, os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')   # never printed
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'kaggle'], check=True)

for ref in ['abdelrahmanelgharibx/mono-kitti', 'ibrahimalobaid/kitte-dataset',
            'klemenko/kitti3dcalib', 'eeemmm1234/kitti-3d-train-calib-label-velodyne']:
    r = subprocess.run(['kaggle', 'datasets', 'files', ref, '--page-size', '12'],
                       text=True, capture_output=True)
    print('\n=====', ref)
    print('\n'.join((r.stdout or r.stderr).splitlines()[:16]))

In [ ]:
import os, glob, time, subprocess, shutil
import numpy as np

assert 'IDX' in globals() and 'DEST' in globals() and 'read_label' in globals(), \
    'The earlier cell variables are gone (runtime reset?). Tell me, and I will give you a self-contained version.'
BASE = '/content/drive/MyDrive/adp_project'
assert os.path.isdir('/content/drive/MyDrive'), 'Drive is not mounted'

have = [i for i in IDX if os.path.exists(f'{DEST}/image_2/{i:06d}.png') and os.path.exists(f'{DEST}/label_2/{i:06d}.txt')]
img_only = [i for i in IDX if os.path.exists(f'{DEST}/image_2/{i:06d}.png') and i not in have]
lab_only = [i for i in IDX if os.path.exists(f'{DEST}/label_2/{i:06d}.txt') and i not in have]
print(f'complete pairs: {len(have)} | image only: {len(img_only)} | label only: {len(lab_only)}')
print('complete pair numbers:', have)
missing = [i for i in IDX if i not in have]
print('first 8 missing numbers:', missing[:8])

objs = [o for i in have for o in read_label(f'{DEST}/label_2/{i:06d}.txt')
        if o['type'] in ('Car', 'Van', 'Truck', 'Pedestrian', 'Cyclist')]
print('usable objects in the pairs we have:', len(objs))

# save what we have to Drive (so a runtime reset cannot lose it)
z = shutil.make_archive(f'{BASE}/data/kitti_depth_subset_partial', 'zip', DEST)
print('saved partial subset to Drive:', z, round(os.path.getsize(z) / 1e6, 1), 'MB')

# raw look at one failing download of each kind (60 s limit)
if missing:
    for ref, remote in ((IMG_REF, f'training/image_2/{missing[0]:06d}.png'),
                        (LAB_REF, f'training/label_2/{missing[0]:06d}.txt')):
        t = time.time()
        try:
            r = subprocess.run(['kaggle', 'datasets', 'download', '-d', ref, '-f', remote,
                                '-p', '/content/kitti_diag', '--unzip'], capture_output=True, text=True, timeout=60)
            print(f'\n{remote} -> exit {r.returncode} after {time.time()-t:.1f}s')
            print('stdout:', (r.stdout or '')[:300].strip())
            print('stderr:', (r.stderr or '')[:400].strip())
        except subprocess.TimeoutExpired:
            print(f'\n{remote} -> TIMED OUT after 60 s')

In [ ]:
import os, glob, time, subprocess, sys, zipfile, shutil
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from google.colab import drive, userdata

drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), 'Drive is not mounted'
BASE = '/content/drive/MyDrive/adp_project'
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')   # never printed
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'kaggle'], check=True)

IMG_REF, LAB_REF = 'ibrahimalobaid/kitte-dataset', 'eeemmm1234/kitti-3d-train-calib-label-velodyne'
DEST = '/content/kitti_sub'
for d in ('image_2', 'label_2'):
    os.makedirs(f'{DEST}/{d}', exist_ok=True)
if len(glob.glob(f'{DEST}/label_2/*.txt')) < 5:                     # /content was wiped: restore from Drive
    zp = f'{BASE}/data/kitti_depth_subset_partial.zip'
    if os.path.exists(zp):
        zipfile.ZipFile(zp).extractall(DEST); print('restored partial subset from Drive')
IDX = list(range(0, 7481, 37))[:200]

def fetch(ref, remote, local_dir, timeout=45):
    base = os.path.basename(remote); final = f'{local_dir}/{base}'
    if os.path.exists(final):
        return True
    for _ in range(2):
        try:
            subprocess.run(['kaggle', 'datasets', 'download', '-d', ref, '-f', remote, '-p', local_dir, '--unzip'],
                           capture_output=True, text=True, timeout=timeout)
        except Exception:
            time.sleep(2); continue
        z = final + '.zip'
        if os.path.exists(z):
            zipfile.ZipFile(z).extractall(local_dir); os.remove(z)
        if not os.path.exists(final):
            hit = glob.glob(f'{local_dir}/**/{base}', recursive=True)
            if hit:
                shutil.move(hit[0], final)
        if os.path.exists(final):
            return True
        time.sleep(2)
    return False

def has_pair(i):
    s = f'{i:06d}'
    return os.path.exists(f'{DEST}/image_2/{s}.png') and os.path.exists(f'{DEST}/label_2/{s}.txt')

def get_pair(i):
    s = f'{i:06d}'
    fetch(IMG_REF, f'training/image_2/{s}.png', f'{DEST}/image_2')
    fetch(LAB_REF, f'training/label_2/{s}.txt', f'{DEST}/label_2')
    return has_pair(i)

def read_label(p):
    objs = []
    for ln in open(p):
        f = ln.split()
        if len(f) >= 15:
            objs.append(dict(type=f[0], trunc=float(f[1]), occ=int(float(f[2])),
                             box=[float(v) for v in f[4:8]], loc=[float(v) for v in f[11:14]]))
    return objs

todo = [i for i in IDX if not has_pair(i)]
print('pairs already on disk:', len(IDX) - len(todo), '| still to fetch:', len(todo))
deadline, t0, empty = time.time() + 600, time.time(), 0
for k in range(0, len(todo), 20):
    if time.time() > deadline:
        print('10-minute limit reached - continuing with what we have'); break
    batch = todo[k:k + 20]
    with ThreadPoolExecutor(2) as ex:
        list(ex.map(get_pair, batch))
    got = sum(has_pair(i) for i in batch)
    empty = empty + 1 if got == 0 else 0
    print(f'batch {k // 20 + 1}: +{got}/{len(batch)} | complete pairs now: {sum(has_pair(i) for i in IDX)} | {time.time() - t0:.0f}s')
    if empty >= 2:
        print('Two empty batches in a row: Kaggle seems to be throttling. Stopping so we do not waste time.'); break
    time.sleep(1)

have = [i for i in IDX if has_pair(i)]
assert len(have) >= 40, f'only {len(have)} complete pairs - send me this output'
types, dist = {}, []
for i in have:
    for o in read_label(f'{DEST}/label_2/{i:06d}.txt'):
        types[o['type']] = types.get(o['type'], 0) + 1
        if o['type'] in ('Car', 'Van', 'Truck', 'Pedestrian', 'Cyclist'):
            dist.append(o['loc'][2])
print('\nframes:', len(have), '| objects by type:', types)
print('forward distance z (m), percentiles 5/25/50/75/95:', np.round(np.percentile(dist, [5, 25, 50, 75, 95]), 1))
os.makedirs(f'{BASE}/data', exist_ok=True)
z = shutil.make_archive(f'{BASE}/data/kitti_depth_subset', 'zip', DEST)
print('saved to Drive:', z, round(os.path.getsize(z) / 1e6, 1), 'MB')

In [ ]:
import os, json, time, zipfile, itertools
import numpy as np, cv2, torch
import matplotlib.pyplot as plt
from google.colab import drive, files

drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), 'Drive is not mounted'
BASE = '/content/drive/MyDrive/adp_project'
DEST = '/content/kitti_eval'
if not os.path.isdir(f'{DEST}/label_2'):
    zp = f'{BASE}/data/kitti_depth_subset.zip'
    assert os.path.exists(zp), 'kitti_depth_subset.zip not found in Drive'
    zipfile.ZipFile(zp).extractall(DEST)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev)
midas = torch.hub.load('intel-isl/MiDaS', 'MiDaS_small', trust_repo=True).to(dev).eval()
MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)

@torch.no_grad()
def depth_map(img_rgb, long_side):
    h, w = img_rgb.shape[:2]
    s = long_side / max(h, w)
    nw = max(32, int(round(w * s / 32)) * 32); nh = max(32, int(round(h * s / 32)) * 32)
    x = cv2.resize(img_rgb, (nw, nh), interpolation=cv2.INTER_CUBIC).astype(np.float32) / 255.0
    x = torch.from_numpy(((x - MEAN) / STD).transpose(2, 0, 1)).unsqueeze(0).to(dev)
    out = midas(x)
    out = torch.nn.functional.interpolate(out.unsqueeze(1), size=(h, w), mode='bicubic', align_corners=False)
    return out.squeeze().cpu().numpy()

def read_label(p):
    res = []
    for ln in open(p):
        f = ln.split()
        if len(f) >= 15:
            res.append(dict(type=f[0], trunc=float(f[1]), occ=int(float(f[2])),
                            box=[float(v) for v in f[4:8]], loc=[float(v) for v in f[11:14]]))
    return res

frames = sorted(int(f[:6]) for f in os.listdir(f'{DEST}/image_2')
                if f.endswith('.png') and os.path.exists(f'{DEST}/label_2/{f[:6]}.txt'))
KEEP = ('Car', 'Van', 'Truck', 'Pedestrian', 'Cyclist')
objs, idx_by_frame = [], {fi: [] for fi in frames}
for fi in frames:
    for o in read_label(f'{DEST}/label_2/{fi:06d}.txt'):
        x1, y1, x2, y2 = o['box']
        if o['type'] in KEEP and (y2 - y1) >= 20 and o['trunc'] <= 0.5 and o['occ'] <= 2 and 3 <= o['loc'][2] <= 80:
            idx_by_frame[fi].append(len(objs))
            objs.append(dict(f=fi, t=o['type'], box=o['box'], Z=o['loc'][2]))
Z = np.array([o['Z'] for o in objs]); F = np.array([o['f'] for o in objs])
half = (F >= np.median(frames)).astype(int)
print(f'frames: {len(frames)} | usable objects: {len(objs)} | by type:',
      {t: sum(o['t'] == t for o in objs) for t in KEEP}, '| fold sizes:', [int((half == h).sum()) for h in (0, 1)])
assert min((half == 0).sum(), (half == 1).sum()) > 30, 'a fold is too small - send me this output'

def region_value(dm, box):
    h, w = dm.shape; x1, y1, x2, y2 = box; bw, bh = x2 - x1, y2 - y1
    xa, xb = int(max(0, x1 + 0.25 * bw)), int(min(w, x2 - 0.25 * bw))
    ya, yb = int(max(0, y1 + 0.4 * bh)), int(min(h, y2))
    return float(np.median(dm[ya:yb, xa:xb])) if (xb - xa >= 2 and yb - ya >= 2) else np.nan

SIZES = [256, 512, 768]
V, T = {}, {}
for sz in SIZES:
    vals, tt = np.full(len(objs), np.nan), []
    for fi in frames:
        img = cv2.cvtColor(cv2.imread(f'{DEST}/image_2/{fi:06d}.png'), cv2.COLOR_BGR2RGB)
        if dev == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter(); dm = depth_map(img, sz)
        if dev == 'cuda': torch.cuda.synchronize()
        tt.append((time.perf_counter() - t0) * 1000)
        for k in idx_by_frame[fi]:
            vals[k] = region_value(dm, objs[k]['box'])
    V[sz], T[sz] = vals, float(np.median(tt[3:]))
    print(f'size {sz}: depth maps done, median {T[sz]:.1f} ms per frame (incl. resize and upsample)')

def fit(v, z):
    ok = np.isfinite(v)
    return np.polyfit(1.0 / z[ok], v[ok], 1)            # v = s*(1/Z) + t
def to_dist(v, s, t):
    d = v - t
    with np.errstate(divide='ignore', invalid='ignore'):
        z = np.where(d > s / 150.0, s / d, 150.0)
    return np.clip(z, 1.0, 150.0)
def crossfit(v):
    P = np.full(len(Z), np.nan)
    for h in (0, 1):
        tr, te = half != h, half == h
        s, t = fit(v[tr], Z[tr])
        P[te] = to_dist(v[te], s, t) if s > 0 else np.nan
    return P
def metrics(Zs, P):
    ok = np.isfinite(P); Zs, P = Zs[ok], P[ok]
    ae = np.abs(Zs - P); ratio = np.maximum(Zs / P, P / Zs)
    return dict(n=int(ok.sum()), mae_m=round(float(ae.mean()), 2), median_abs_rel_err_pct=round(100 * float(np.median(ae / Zs)), 1),
                within25_pct=round(100 * float((ratio < 1.25).mean()), 1), bias_m=round(float((P - Zs).mean()), 2))
def pair_order(v):
    good = tot = 0
    for fi in frames:
        ks = [k for k in idx_by_frame[fi] if np.isfinite(v[k])]
        for a, b in itertools.combinations(ks, 2):
            if max(Z[a], Z[b]) / min(Z[a], Z[b]) < 1.2: continue
            tot += 1; good += bool((v[a] > v[b]) == (Z[a] < Z[b]))
    return round(100 * good / max(tot, 1), 1), tot

res = {'frames': len(frames), 'objects': len(objs), 'sizes': {}}
P_by = {}
for sz in SIZES:
    P = crossfit(V[sz]); P_by[sz] = P
    po, npairs = pair_order(V[sz])
    res['sizes'][sz] = dict(metrics=metrics(Z, P), pair_order_acc_pct=po, pairs=npairs, ms_per_frame=round(T[sz], 1))
Pc = np.full(len(Z), np.nan)
for h in (0, 1):
    Pc[half == h] = np.median(Z[half != h])
yb = np.array([o['box'][3] for o in objs])
with np.errstate(divide='ignore', invalid='ignore'):
    Pg = np.where(yb > 172.854 + 2, 721.5377 * 1.65 / (yb - 172.854), np.nan)
Pg = np.clip(Pg, 1.0, 150.0)
res['baseline_always_median'] = metrics(Z, Pc)
res['baseline_ground_plane_assumed_kitti_calibration'] = metrics(Z, Pg)

print('\n=== results (every object scored by a conversion fitted on the OTHER half) ===')
for sz in SIZES:
    r = res['sizes'][sz]; print(f'MiDaS_small long side {sz}:', r['metrics'], '| near/far order correct:', r['pair_order_acc_pct'], '% of', r['pairs'], 'pairs')
print('baseline: always predict the typical distance:', res['baseline_always_median'])
print('baseline: flat-road geometry (assumed KITTI calibration):', res['baseline_ground_plane_assumed_kitti_calibration'])

best = min(SIZES, key=lambda s: res['sizes'][s]['metrics']['median_abs_rel_err_pct'])
print('\nbest input size on this data (selected on the same frames):', best)
bins = [(3, 15), (15, 30), (30, 50), (50, 80.01)]
res['best_size'] = best; res['by_distance_bin'] = {}
for lo, hi in bins:
    m = (Z >= lo) & (Z < hi)
    if m.sum() >= 5:
        res['by_distance_bin'][f'{lo}-{int(hi)} m'] = metrics(Z[m], P_by[best][m])
        print(f'{lo}-{int(hi)} m:', res['by_distance_bin'][f'{lo}-{int(hi)} m'])
s_all, t_all = fit(V[best], Z)
res['fit_on_all_frames'] = dict(size=best, s=float(s_all), t=float(t_all),
                                note='depth_value = s*(1/Z) + t; valid only for KITTI-like cameras')
res['notes'] = ('GT 2D boxes used, not detections; 140 frames; sizes compared on the same data; '
                'ground-plane baseline uses assumed typical KITTI calibration; KITTI licence terms unconfirmed')
os.makedirs(f'{BASE}/logs', exist_ok=True)
json.dump(res, open(f'{BASE}/logs/midas_kitti_eval.json', 'w'), indent=1)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ok = np.isfinite(P_by[best])
ax[0].scatter(Z[ok], P_by[best][ok], s=8, alpha=0.5); ax[0].plot([0, 80], [0, 80], 'k--', lw=1)
ax[0].set_xlabel('true distance (m)'); ax[0].set_ylabel('estimated distance (m)'); ax[0].set_xlim(0, 80); ax[0].set_ylim(0, 100)
ax[0].set_title(f'MiDaS_small (long side {best}) vs KITTI truth\nheld-out halves, n={int(ok.sum())}')
names = list(res['by_distance_bin']); vals = [res['by_distance_bin'][k]['median_abs_rel_err_pct'] for k in names]
ax[1].bar(names, vals, color='#2b6cb0')
for i, k in enumerate(names):
    ax[1].text(i, vals[i] + 0.5, f"n={res['by_distance_bin'][k]['n']}", ha='center', fontsize=8)
ax[1].set_ylabel('median absolute relative error (%)'); ax[1].set_title('error by true distance')
plt.tight_layout(); plt.savefig('/content/midas_kitti_eval.png', dpi=110); plt.show()
files.download('/content/midas_kitti_eval.png'); files.download(f'{BASE}/logs/midas_kitti_eval.json')

In [ ]:
import os, sys, json, time, subprocess
import numpy as np, cv2, torch
from google.colab import drive, files

drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), 'Drive is not mounted'
BASE = '/content/drive/MyDrive/adp_project'
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'ultralytics', 'lapx', 'timm'], check=True)
from ultralytics import YOLO

ev = json.load(open(f'{BASE}/logs/midas_kitti_eval.json'))['fit_on_all_frames']
S_, T_, LS = ev['s'], ev['t'], ev['size']
print('conversion from KITTI fit: s=%.1f t=%.2f at long side %d' % (S_, T_, LS))
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
midas = torch.hub.load('intel-isl/MiDaS', 'MiDaS_small', trust_repo=True).to(dev).eval()
MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)

@torch.no_grad()
def depth_map(img_rgb, long_side):
    h, w = img_rgb.shape[:2]; s = long_side / max(h, w)
    nw = max(32, int(round(w * s / 32)) * 32); nh = max(32, int(round(h * s / 32)) * 32)
    x = cv2.resize(img_rgb, (nw, nh), interpolation=cv2.INTER_CUBIC).astype(np.float32) / 255.0
    x = torch.from_numpy(((x - MEAN) / STD).transpose(2, 0, 1)).unsqueeze(0).to(dev)
    out = midas(x)
    out = torch.nn.functional.interpolate(out.unsqueeze(1), size=(h, w), mode='bicubic', align_corners=False)
    return out.squeeze().cpu().numpy()

def region_value(dm, box):
    h, w = dm.shape; x1, y1, x2, y2 = box; bw, bh = x2 - x1, y2 - y1
    xa, xb = int(max(0, x1 + 0.25 * bw)), int(min(w, x2 - 0.25 * bw))
    ya, yb = int(max(0, y1 + 0.4 * bh)), int(min(h, y2))
    return float(np.median(dm[ya:yb, xa:xb])) if (xb - xa >= 2 and yb - ya >= 2) else np.nan

def to_dist(v):
    d = v - T_
    return float(np.clip(S_ / d, 1.0, 150.0)) if (np.isfinite(v) and d > S_ / 150.0) else 150.0

W = f'{BASE}/runs/yolo11s_bdd/weights/best.pt'
assert os.path.exists(W), 'best.pt not found'
VEH = [0, 1, 2, 3, 4, 6, 7]
OUTW, MAX_SEC = 1280, 12
OUTD = f'{BASE}/outputs/depth_demo'; os.makedirs(OUTD, exist_ok=True)
stats, dl = {}, []

for name in ('clip2', 'clip4'):
    src = f'{BASE}/videos/{name}.mp4'
    assert os.path.exists(src), src + ' missing'
    cap = cv2.VideoCapture(src); fps = cap.get(cv2.CAP_PROP_FPS) or 30
    ow0, oh0 = int(cap.get(3)), int(cap.get(4))
    OW = OUTW; OH = int(round(oh0 * OW / ow0 / 2) * 2); limit = int(fps * MAX_SEC)
    raw = f'/content/{name}_depth_raw.mp4'
    vw = cv2.VideoWriter(raw, cv2.VideoWriter_fourcc(*'mp4v'), fps, (OW, OH))
    yolo = YOLO(W); ema, seen, ms, n, mid = {}, set(), [], 0, None
    while n < limit:
        ok, fr = cap.read()
        if not ok: break
        fr = cv2.resize(fr, (OW, OH), interpolation=cv2.INTER_AREA)
        if dev == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        r = yolo.track(fr, persist=True, tracker='bytetrack.yaml', conf=0.35, imgsz=640, classes=VEH, verbose=False)[0]
        dm = depth_map(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB), LS)
        out = fr.copy()
        if r.boxes is not None and r.boxes.id is not None:
            for b, tid, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.id.int().tolist(), r.boxes.cls.int().tolist()):
                d = to_dist(region_value(dm, b)); e = ema.get(tid)
                ema[tid] = d if e is None else 0.7 * e + 0.3 * d; seen.add(tid)
                x1, y1, x2, y2 = [int(v) for v in b]
                cv2.rectangle(out, (x1, y1), (x2, y2), (0, 220, 0), 2)
                lab = f'{r.names[c]} {tid} ~{ema[tid]:.0f} m'
                (tw, th), _ = cv2.getTextSize(lab, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
                cv2.rectangle(out, (x1, max(0, y1 - th - 6)), (x1 + tw + 4, y1), (0, 220, 0), -1)
                cv2.putText(out, lab, (x1 + 2, max(th, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 1)
        p2, p98 = np.percentile(dm, [2, 98])
        dv = cv2.applyColorMap((np.clip((dm - p2) / max(p98 - p2, 1e-6), 0, 1) * 255).astype(np.uint8), cv2.COLORMAP_INFERNO)
        iw, ih = OW // 4, OH // 4
        out[8:8 + ih, OW - iw - 8:OW - 8] = cv2.resize(dv, (iw, ih))
        cv2.putText(out, 'MiDaS depth', (OW - iw - 4, 8 + ih + 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.putText(out, 'distance: approx. (MiDaS + KITTI-fitted conversion; this camera is NOT calibrated)',
                    (10, OH - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        if dev == 'cuda': torch.cuda.synchronize()
        ms.append((time.perf_counter() - t0) * 1000)
        vw.write(out); n += 1
        if n == limit // 2: mid = out.copy()
    cap.release(); vw.release()
    final = f'{OUTD}/{name}_depth_demo.mp4'
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', raw, '-vcodec', 'libx264', '-pix_fmt', 'yuv420p', final], check=True)
    jpg = f'/content/{name}_depth_frame.jpg'
    cv2.imwrite(jpg, mid if mid is not None else out, [cv2.IMWRITE_JPEG_QUALITY, 88]); dl.append(jpg)
    med = float(np.median(ms[3:])) if len(ms) > 3 else float(np.median(ms))
    stats[name] = dict(frames=n, seconds=round(n / fps, 1), unique_tracks=len(seen),
                       pipeline_ms_per_frame_t4=round(med, 1), pipeline_fps_t4=round(1000 / med, 1),
                       note='YOLO11s + ByteTrack + MiDaS_small(512) + drawing, excludes video decode/encode; no ground truth for these clips')
    print(name, stats[name])

json.dump(stats, open(f'{BASE}/logs/depth_demo_stats.json', 'w'), indent=1)
dl.append(f'{BASE}/logs/depth_demo_stats.json')
for f in dl: files.download(f)
print('videos in Drive:', sorted(os.listdir(OUTD)))